# Эксперимент: LightGBM в ансамбле

Поверх v5. Ансамбль из двух сидов CatBoost дал -0.002 score на сезонных фолдах; второй сид той же модели ошибается почти так же, как первый. Модели другого семейства ошибаются по-другому - у победителя M5 ансамбль LightGBM-моделей. Участник `{"library": "lightgbm"}` (`src/lgbm.py`) учится на тех же признаках и строках, с тем же весом свежих недель: классификатор (binary) и регрессор (log1p). Вероятности усредняются, количества - в логарифмах.

Варианты: v5 + LightGBM (три участника), один CatBoost + LightGBM, только LightGBM (для сравнения), v5 + LightGBM с порогом 0.55 (среднее трёх моделей может иначе калибровать вероятности).

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_lgbm"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; первый - v5, с ним и сравниваем.

In [2]:
LGB = {"library": "lightgbm"}
VARIANTS = {
    "v5": V5,
    "v5 + LightGBM": {**V5, "ensemble": [{}, {"random_seed": 7}, LGB]},
    "CatBoost + LightGBM": {**V5, "ensemble": [{}, LGB]},
    "только LightGBM": {**V5, "ensemble": [LGB]},
}
VARIANTS["v5 + LightGBM, порог 0.55"] = {**VARIANTS["v5 + LightGBM"], "threshold": 0.55}
list(VARIANTS)

['v5',
 'v5 + LightGBM',
 'CatBoost + LightGBM',
 'только LightGBM',
 'v5 + LightGBM, порог 0.55']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2025 готов


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2024 готов


recent готов


bias                         rmse             \
фолд                      recent season2024 season2025 recent season2024   
вариант                                                                    
CatBoost + LightGBM        75.51      87.25      80.45   3.35       1.86   
v5                         76.52      86.37      81.08   3.35       1.93   
v5 + LightGBM              75.69      86.59      80.59   3.34       1.88   
v5 + LightGBM, порог 0.55  75.71      86.03      81.02   3.35       1.89   
только LightGBM            74.88      87.54      79.88   3.38       1.83   

                                      smape                         wape  \
фолд                      season2025 recent season2024 season2025 recent   
вариант                                                                    
CatBoost + LightGBM             2.66  58.57      42.99      57.55  47.04   
v5                              2.64  58.71      42.83      57.52  47.10   
v5 + LightGBM                   2.64  58.60      42.87      57.47  47.00   
v5 + LightGBM, порог 0.55       2.65  58.24      42.73      57.53  46.97   
только LightGBM                 2.68  58.69      43.31      57.58  47.26   

                                                 
фолд                      season2024 season2025  
вариант                                          
CatBoost + LightGBM            62.57      51.17  
v5                             62.89      51.18  
v5 + LightGBM                  62.61      51.07  
v5 + LightGBM, порог 0.55      62.54      51.14  
только LightGBM                62.70      51.25

## Выбор

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["v5"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["v5", "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print("v5 в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 в этом запуске: {'season2025': 0.8089, 'season2024': 0.8236, 'recent': 0.7871} | по сезону 0.8163
лучший вариант: v5 + LightGBM {'season2025': 0.8076, 'season2024': 0.8205, 'recent': 0.785, 'по сезону': 0.814}


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
v5 + LightGBM,0.8076,0.8205,0.7850,0.8140,True
"v5 + LightGBM, порог 0.55",0.8093,0.8192,0.7832,0.8142,False
CatBoost + LightGBM,0.8091,0.8205,0.7853,0.8148,False
только LightGBM,0.8099,0.8224,0.7871,0.8162,False
v5,0.8089,0.8236,0.7871,0.8163,False
